In [19]:
# Uninstall potentially problematic packages
!pip uninstall -y realesrgan basicsr torchaudio

# Install the latest compatible basicsr directly from its GitHub repository
!pip install -q git+https://github.com/xinntao/basicsr.git

# Reinstall realesrgan and other dependencies (torch/torchvision will remain as Colab's default compatible versions)
!pip install -q realesrgan gfpgan opencv-python-headless

Found existing installation: realesrgan 0.3.0
Uninstalling realesrgan-0.3.0:
  Successfully uninstalled realesrgan-0.3.0
Found existing installation: basicsr 1.4.2
Uninstalling basicsr-1.4.2:
  Successfully uninstalled basicsr-1.4.2
  Preparing metadata (setup.py) ... done


In [ ]:
import torch
import torchvision

print(f"Torch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

Torch version: 2.10.0+cu128
Torchvision version: 0.25.0+cu128


# Real-ESRGAN 4K Video Upscaler (Anime Model)

**Before running:**
1. Go to **Runtime → Change runtime type → GPU** (T4 is fine)
2. Upload your 720p MP4 to Google Drive
3. Set the input/output paths in the **Configuration** cell below

In [ ]:
import subprocess
import json
import time # Import time for logging
import os

FRAMES_DIR = "/content/frames_input"
FRAMES_UP_DIR = "/content/frames_upscaled"
os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(FRAMES_UP_DIR, exist_ok=True)

# Get video info
probe = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_streams", INPUT_VIDEO],
    capture_output=True, text=True
)
info = json.loads(probe.stdout)
video_stream = next(s for s in info["streams"] if s["codec_type"] == "video")
fps_parts = video_stream["r_frame_rate"].split("/")
FPS = round(int(fps_parts[0]) / int(fps_parts[1]), 3)
src_w, src_h = int(video_stream["width"]), int(video_stream["height"])
print(f"Source: {src_w}x{src_h} @ {FPS} fps")

# Get total number of frames for better logging
total_frames_expected = None
try:
    # Try to get nb_frames directly from ffprobe
    total_frames_expected = int(video_stream.get('nb_frames', 0))
    if total_frames_expected == 0 and 'format' in info and 'duration' in info['format']:
        # Fallback: estimate from duration and FPS if nb_frames is not present
        duration = float(info['format']['duration'])
        total_frames_expected = int(duration * FPS)
except (KeyError, ValueError):
    pass # Cannot get an accurate estimate

# Extract frames
print("Extracting frames...")
if total_frames_expected:
    print(f"Expecting approximately {total_frames_expected} frames.")

start_extract_time = time.time()
!ffmpeg -y -i "{INPUT_VIDEO}" -qscale:v 2 "{FRAMES_DIR}/frame_%06d.png" -loglevel warning
end_extract_time = time.time()

total_frames = len([f for f in os.listdir(FRAMES_DIR) if f.endswith(".png")])
print(f"\u2713 Extracted {total_frames} frames in {(end_extract_time - start_extract_time):.2f} seconds")

# 5. Upscale Frames with Real-ESRGAN (Anime Model)
# This is the slow step. Progress is printed every 50 frames. If the session disconnects, re-run — it will skip already-upscaled frames.
import cv2
import numpy as np
# time is already imported above
from realesrgan import RealESRGANer
from basicsr.archs.srvgg_arch import SRVGGNetCompact

# Build the anime video v3 model (uses SRVGGNetCompact, NOT RRDBNet)
model = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64,
                        num_conv=16, upscale=4, act_type='prelu')
upsampler = RealESRGANer(
    scale=4,
    model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth",
    model=model, # Pass the instantiated model here
    tile=TILE_SIZE,
    tile_pad=10,
    pre_pad=0,
    half=True,  # FP16 for speed on Colab GPUs
)
print("\u2713 Real-ESRGAN anime model loaded")

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith(".png"))
already_done = set(os.listdir(FRAMES_UP_DIR))
to_process = [f for f in frames if f not in already_done]
print(f"Frames to upscale: {len(to_process)} ({len(already_done)} already done)")

start = time.time()
for i, fname in enumerate(to_process):
    img = cv2.imread(os.path.join(FRAMES_DIR, fname), cv2.IMREAD_COLOR)
    output, _ = upsampler.enhance(img, outscale=4)

    # Resize to exact target resolution
    if output.shape[1] != OUTPUT_WIDTH or output.shape[0] != OUTPUT_HEIGHT:
        output = cv2.resize(output, (OUTPUT_WIDTH, OUTPUT_HEIGHT),
                            interpolation=cv2.INTER_LANCZOS4)

    cv2.imwrite(os.path.join(FRAMES_UP_DIR, fname), output)

    if (i + 1) % 50 == 0 or (i + 1) == len(to_process):
        elapsed = time.time() - start
        fps_rate = (i + 1) / elapsed
        remaining = (len(to_process) - i - 1) / fps_rate
        print(f"  [{i+1}/{len(to_process)}] {fps_rate:.2f} frames/sec \u2014 "
              f"~{remaining/60:.0f} min remaining")

print(f"\n\u2713 Upscaling complete in {(time.time()-start)/60:.1f} minutes")

# 6. Reassemble Upscaled Frames into 4K Video
# Build concat list (handles any gaps gracefully)
up_frames = sorted(f for f in os.listdir(FRAMES_UP_DIR) if f.endswith(".png"))
concat_path = "/content/concat_list.txt"
frame_dur = f"{1/FPS:.10f}"
with open(concat_path, "w") as f:
    for fname in up_frames:
        f.write(f"file '{FRAMES_UP_DIR}/{fname}'\n")
        f.write(f"duration {frame_dur}\n")

print(f"Encoding {len(up_frames)} frames \u2192 {OUTPUT_VIDEO}")
print(f"Settings: {OUTPUT_WIDTH}x{OUTPUT_HEIGHT}, {FPS} fps, H.264 CRF {CRF}")

# Encode video with audio from original
!ffmpeg -y \
  -f concat -safe 0 -i "{concat_path}" \
  -i "{INPUT_VIDEO}" \
  -map 0:v:0 -map 1:a:0? \
  -c:v h264_nvenc -crf {CRF} \
  -pix_fmt yuv420p \
  -c:a aac -b:a 320k \
  -r {FPS} \
  -movflags +faststart \
  "{OUTPUT_VIDEO}" \
  -loglevel warning -stats

size_mb = os.path.getsize(OUTPUT_VIDEO) / (1024 * 1024)
print(f"\n\u2713 Done! Saved to: {OUTPUT_VIDEO} ({size_mb:.1f} MB)")

# 7. Cleanup (Optional)
# Delete the temporary frame directories from Colab to free disk space.
import shutil
shutil.rmtree(FRAMES_DIR, ignore_errors=True)
shutil.rmtree(FRAMES_UP_DIR, ignore_errors=True)
if os.path.exists(concat_path):
    os.remove(concat_path)
print("\u2713 Temporary files cleaned up")

Source: 1280x720 @ 29.97 fps
Extracting frames...
Expecting approximately 7080 frames.
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x570735a384c0] stream 0, timescale not set
Guessed Channel Layout for Input Stream #0.0 : stereo
✓ Extracted 7080 frames in 1125.03 seconds


RuntimeError: Error(s) in loading state_dict for RRDBNet:
	Missing key(s) in state_dict: "conv_first.weight", "conv_first.bias", "body.0.rdb1.conv1.weight", "body.0.rdb1.conv1.bias", "body.0.rdb1.conv2.weight", "body.0.rdb1.conv2.bias", "body.0.rdb1.conv3.weight", "body.0.rdb1.conv3.bias", "body.0.rdb1.conv4.weight", "body.0.rdb1.conv4.bias", "body.0.rdb1.conv5.weight", "body.0.rdb1.conv5.bias", "body.0.rdb2.conv1.weight", "body.0.rdb2.conv1.bias", "body.0.rdb2.conv2.weight", "body.0.rdb2.conv2.bias", "body.0.rdb2.conv3.weight", "body.0.rdb2.conv3.bias", "body.0.rdb2.conv4.weight", "body.0.rdb2.conv4.bias", "body.0.rdb2.conv5.weight", "body.0.rdb2.conv5.bias", "body.0.rdb3.conv1.weight", "body.0.rdb3.conv1.bias", "body.0.rdb3.conv2.weight", "body.0.rdb3.conv2.bias", "body.0.rdb3.conv3.weight", "body.0.rdb3.conv3.bias", "body.0.rdb3.conv4.weight", "body.0.rdb3.conv4.bias", "body.0.rdb3.conv5.weight", "body.0.rdb3.conv5.bias", "body.1.rdb1.conv1.weight", "body.1.rdb1.conv1.bias", "body.1